# Snowflake: Complete Introduction
## D380_SnowFlakeIntroduction

**Storage • Query processing • Virtual warehouses • Compute pools • Elastic scaling**

This Markdown-only lesson introduces Snowflake for learners who know basic SQL and data warehouse concepts. No account, credentials, packages, or running compute are required to read it. SQL blocks are illustrative text, not executable notebook cells.

**Learning outcomes:** explain where Snowflake fits in a data platform; trace how data is stored and queried; distinguish storage from compute; select warehouse sizes; explain scaling and cost; distinguish compute pools from SQL warehouses.

Current product details and stock listing checked on **7 September 2026**. Feature availability and consumption rates depend on edition, cloud, region, and warehouse generation. Official references appear beside each topic.

## 1. What is Snowflake?

Snowflake is a managed cloud data platform for storing, processing, analyzing, and sharing data. A common starting use case is a SQL data warehouse: combine operational data into reliable tables for reporting and analytics.

Snowflake runs on supported AWS, Microsoft Azure, and Google Cloud infrastructure. Customers manage data, SQL, permissions, and workload configuration; Snowflake manages the underlying platform infrastructure. An account belongs to a particular cloud and region; using a multi-cloud product does not automatically distribute every account across all clouds.

Its architectural distinction is **independent storage and compute**. Stored tables survive when query compute is suspended. Multiple compute resources can access the same governed data.

Source: [Key concepts and architecture](https://docs.snowflake.com/en/user-guide/intro-key-concepts).

## 2. When did Snowflake start, and where is its stock listed?

| Milestone | Note |
|---|---|
| Founded | **2012** |
| Public-market debut | **16 September 2020**, on the New York Stock Exchange |
| Current listing | **NYSE: SNOW**, checked on 7 September 2026 |

The company name is Snowflake Inc. The stock ticker identifies the company in financial markets; it is unrelated to a Snowflake account identifier or database name.

Sources: [Snowflake company facts](https://www.snowflake.com/resource/vous-pilotez-strategie-multi-cloud/snowflake-fast-facts-2-2-2/), [IPO announcement](https://www.snowflake.com/en/news/press-releases/snowflake-announces-pricing-of-initial-public-offering/), [current SEC filing: exchange and ticker](https://www.sec.gov/Archives/edgar/data/1640147/000164014726000037/snow-20260731.htm).

## 3. Where Snowflake fits in a data engineering stack

An illustrative retail architecture:

```text
Operational databases     Applications / APIs     Event streams
          \                       |                    /
               Ingestion and orchestration
                          |
              Object storage / landing files
                          |
                Snowflake RAW tables
                          |
           SQL / Snowpark transformations
                          |
           CURATED tables -> ANALYTICS marts
                          |
              BI reports / analysts / apps
```

The file landing step is optional: streaming ingestion can deliver rows without a file-based workflow. A team might use orchestration tools to coordinate dependencies and a BI tool to visualize results.

**ETL** transforms data before loading the destination. **ELT** loads first and transforms using destination compute. Snowflake often supports ELT because SQL transformations execute close to the stored data.

Example: a retailer loads orders, cleans invalid records, joins customer details, and publishes daily sales. Source applications still own order entry; the analytical platform produces historical business insight.

Source: [Data loading overview](https://docs.snowflake.com/en/user-guide/data-load-overview).

## 4. The vocabulary: database, schema, table, and warehouse

| Term | Meaning | Teaching example |
|---|---|---|
| Organization | Groups related Snowflake accounts | A global enterprise |
| Account | Administrative environment in a cloud/region | Training account |
| Database | Logical container for schemas | `RETAIL_DB` |
| Schema | Groups related database objects | `RAW`, `CURATED`, `ANALYTICS` |
| Table | Stores rows and columns | `ORDERS` |
| View | Named query presenting data | `DAILY_SALES_V` |
| Virtual warehouse | Compute resource that runs SQL work | `BI_WH` |
| Role | Collection of access privileges | `ANALYST_ROLE` |

A fully qualified table name looks like `RETAIL_DB.RAW.ORDERS`. The warehouse is selected independently of that name.

**Data warehouse** describes an analytical data system. A Snowflake **virtual warehouse** is the compute engine. Increasing virtual warehouse size does not increase a database's storage limit.

Sources: [Architecture](https://docs.snowflake.com/en/user-guide/intro-key-concepts), [Access control](https://docs.snowflake.com/en/user-guide/security-access-control-overview).

## 5. Architecture: three cooperating layers

```text
        Clients: Snowsight, SQL drivers, BI tools, applications
                                |
        CLOUD SERVICES: authentication, metadata, optimization
                                |
        COMPUTE:   INGEST_WH    TRANSFORM_WH    BI_WH
                       \            |           /
        STORAGE: centrally accessible persistent table data
```

| Layer | Responsibility |
|---|---|
| Storage | Durable data in cloud storage |
| Compute | Parallel execution using virtual warehouses |
| Cloud services | Coordination, access checking, metadata, and query planning |

The architecture combines shared access to persistent data with distributed computation inside each warehouse. Separating these layers enables teams to tune their workloads without copying every table into a separate compute cluster.

Source: [Snowflake architecture](https://docs.snowflake.com/en/user-guide/intro-key-concepts).

## 6. How native Snowflake tables store data

For standard native tables, Snowflake organizes data automatically into **micro-partitions**, using compressed columnar storage. Each micro-partition represents roughly **50–500 MB of uncompressed data**; the bytes physically stored are compressed.

Columnar storage groups values by column. A sales query reading `ORDER_DATE` and `AMOUNT` can avoid reading unrelated columns such as delivery instructions.

Snowflake records metadata such as column value ranges and distinct-value information. It uses this information to eliminate irrelevant micro-partitions during a query, called **partition pruning**.

Changes produce new micro-partition versions rather than editing the existing immutable units in place. Retained historical versions contribute to storage consumption.

Source: [Micro-partitions and clustering](https://docs.snowflake.com/en/user-guide/tables-clustering-micropartitions).

## 7. Pruning and clustering: an example

Imagine this simplified layout; real partitions contain many columns and richer metadata.

| Micro-partition | Minimum order date | Maximum order date |
|---|---|---|
| A | 2026-01-01 | 2026-01-10 |
| B | 2026-02-01 | 2026-02-10 |
| C | 2026-03-01 | 2026-03-10 |

```sql
SELECT SUM(amount)
FROM retail_db.raw.orders
WHERE order_date = '2026-02-05'::DATE;
```

The date ranges let Snowflake skip A and C. If every partition overlaps every month, this filter prunes less effectively.

Clustering describes how related values are colocated. Selected large tables can benefit from a clustering key, but maintaining clustering consumes resources. Choose it after examining real filtering patterns and query profiles. Snowflake's automatic micro-partitioning does not mean every table needs a user-defined clustering key.

Source: [Micro-partitions and clustering](https://docs.snowflake.com/en/user-guide/tables-clustering-micropartitions).

## 8. Native tables, Iceberg, and data formats

The native-table storage explanation is not universal for every Snowflake table type. **Apache Iceberg tables** use an open table format and data files in external cloud storage. Snowflake supports different catalog and management arrangements; responsibility for files and maintenance depends on the arrangement.

An Iceberg design can help when several engines need access to a lake's open-format tables. A native-table design gives Snowflake control over its internal storage organization. Select based on interoperability, governance, operations, and workload needs.

Source: [Apache Iceberg tables](https://docs.snowflake.com/en/user-guide/tables-iceberg).

Loading also separates **file format** from **table storage format**. CSV, JSON, and Parquet are examples of ingestion formats; loading a CSV into a native table does not leave that table as a CSV file. Semi-structured values can be represented using `VARIANT`, with `OBJECT` and `ARRAY` for nested structures.

Source: [Data loading overview](https://docs.snowflake.com/en/user-guide/data-load-overview).

## 9. How a SQL query is processed

For an ordinary analytical query that is not satisfied by result reuse:

1. A client submits SQL with an active role and session context.
2. Snowflake checks access and resolves the referenced objects.
3. The optimizer builds an execution plan using metadata.
4. The selected warehouse provides execution resources, resuming if configured.
5. Workers read needed data, apply filters, join rows, and calculate aggregates.
6. Intermediate data may move between workers; final results return to the client.

**MPP — massively parallel processing** — divides work across compute resources. A large aggregation can calculate partial totals in parallel and combine them. Joins and sorts can require redistribution of intermediate data, so parallelism has communication costs.

An execution plan determines how work is distributed; SQL users normally specify the desired result, not the worker-level implementation.

Source: [Snowflake architecture paper](https://www.snowflake.com/wp-content/uploads/2019/06/Snowflake_SIGMOD.pdf).

## 10. Caching: why repeated queries can be faster

| Mechanism | What it helps avoid | Important boundary |
|---|---|---|
| Persisted query results | Re-executing an eligible query | Reuse depends on unchanged data and other eligibility checks |
| Warehouse data cache | Re-reading table data from remote storage | Local to warehouse compute; lost when suspended |
| Metadata | Unnecessary data reads and planning work | Metadata alone cannot answer every query |

Persisted results are normally retained for 24 hours. An identical-looking query is not guaranteed to reuse results: privileges, data changes, functions, and other conditions matter.

Source: [Persisted query results](https://docs.snowflake.com/en/user-guide/querying-persisted-results).

Auto-suspension saves idle compute credits but discards the warehouse cache. A first query after resume can therefore be slower than a warm-cache query. In performance comparisons, distinguish result reuse, warm data cache, and cold execution; otherwise the benchmark may measure caching instead of warehouse capacity.

Source: [Warehouse cache](https://docs.snowflake.com/en/user-guide/performance-query-warehouse-cache).

## 11. Virtual warehouses: what you actually size

A virtual warehouse supplies CPU, memory, and temporary storage for query execution and operations such as SQL transformations and bulk loading. It does not own the permanent tables it accesses.

For an illustrative team:

| Warehouse | Workload | Reason for separation |
|---|---|---|
| `INGEST_WH` | Scheduled batch loads | Loading gets its own compute |
| `TRANSFORM_WH` | Large joins and aggregations | Transformation capacity can be tuned independently |
| `BI_WH` | Dashboard queries | Reporting is less exposed to competing ETL compute |
| `DEV_WH` | Learning and experimentation | Development has an independent resource budget |

These resources can read the same authorized tables. Separate warehouses provide compute isolation; transactional conflicts or upstream data delays can still affect workloads.

Source: [Virtual warehouses](https://docs.snowflake.com/en/user-guide/warehouses).

## 12. Warehouse sizes and relative consumption

The table below applies to **Gen1 standard warehouses, per running cluster**. These are credits, not dollar prices or fixed CPU specifications.

| Size | SQL size value | Credits/hour |
|---|---|---:|
| X-Small | `XSMALL` | 1 |
| Small | `SMALL` | 2 |
| Medium | `MEDIUM` | 4 |
| Large | `LARGE` | 8 |
| X-Large | `XLARGE` | 16 |
| 2X-Large | `XXLARGE` | 32 |
| 3X-Large | `XXXLARGE` | 64 |
| 4X-Large | `X4LARGE` | 128 |
| 5X-Large | `X5LARGE` | 256 |
| 6X-Large | `X6LARGE` | 512 |

Each size step doubles the Gen1 credit rate. Larger sizes provide more resources, but do not guarantee twice the speed. Large-size availability varies by cloud/region. Warehouse size specifies computation capacity, not stored terabytes.

Source: [Warehouse sizes and billing](https://docs.snowflake.com/en/user-guide/warehouses-overview).

## 13. Warehouse type and generation also matter

**Standard warehouses** support general SQL workloads. **Snowpark-optimized warehouses** provide configurations intended for memory-intensive Snowpark workloads. Choose according to workload behavior, not the programming language alone.

Source: [Virtual warehouses](https://docs.snowflake.com/en/user-guide/warehouses).

**Generation 2 standard warehouses** use newer execution infrastructure. Their availability and credit rates must be checked separately. The Gen1 size table must not be treated as the price list for Gen2, Snowpark-optimized resources, compute pools, or serverless features.

For a benchmark, record warehouse type, generation, size, cluster count, cache conditions, query runtime, and actual credits. A size label alone is insufficient for a fair comparison.

Source: [Generation 2 warehouses](https://docs.snowflake.com/en/user-guide/warehouses-gen2).

## 14. Scaling up: more resources per cluster

**Scale up** means choosing a larger warehouse size; **scale down** means choosing a smaller size.

```text
One Medium cluster  ->  One Large cluster
        More compute resources per cluster
```

This can help a large query with substantial parallel work or memory pressure. It may do little for tiny queries, inefficient SQL, or a load constrained by file layout. Increasing size does not guarantee proportional performance improvement.

Use Query Profile to understand bottlenecks before resizing. Compare elapsed time and credits for the same representative workload. A faster execution can cost less, the same, or more depending on how much time it saves.

Source: [Warehouse considerations](https://docs.snowflake.com/en/user-guide/warehouses-considerations).

Resizing a running warehouse makes additional resources available to queued or new queries once provisioned; it does not speed up an already-running query by adding workers to it.

Source: [Warehouse overview](https://docs.snowflake.com/en/user-guide/warehouses-overview).

## 15. Scaling out: more clusters for concurrency

**Scale out** adds clusters to a multi-cluster warehouse; **scale in** removes them. This primarily addresses many simultaneous queries and queuing. It does not combine all clusters into one giant execution cluster for a single query.

```text
Low demand:   BI_WH [Medium cluster 1]
Peak demand:  BI_WH [Medium cluster 1] [Medium cluster 2] [Medium cluster 3]
```

Multi-cluster warehouses require Enterprise Edition or higher.

| Setting | Meaning |
|---|---|
| Minimum clusters | Baseline while the warehouse runs |
| Maximum clusters | Upper bound for scale-out |
| Auto-scale mode | Minimum is lower than maximum |
| Maximized mode | Minimum equals maximum; configured clusters run together |
| Standard policy | Favors responsiveness and reducing queues |
| Economy policy | Favors fuller utilization and may tolerate more queuing |

Cluster limits depend on warehouse size and supported configuration; do not assume ten is a universal SQL limit. Snowflake adds and removes clusters within configured bounds as demand changes.

Source: [Multi-cluster warehouses](https://docs.snowflake.com/en/user-guide/warehouses-multicluster).

## 16. Elasticity, auto-suspend, and auto-resume

**Elasticity** means adjusting resource use as demand changes. It involves several independent choices:

| Demand change | Response |
|---|---|
| More stored data | Persistent storage grows independently |
| Heavier analytical queries | Benchmark a larger warehouse |
| More simultaneous dashboard users | Consider additional clusters |
| Independent teams | Assign separate warehouses |
| No query activity | Suspend idle warehouse compute |
| Work returns | Resume automatically if configured |

For standard warehouses, billing is per second with a **60-second minimum each time compute starts**. Idle time before suspension is billable. Suspending compute does not remove stored data or stop storage charges.

Auto-resume is not the same as resizing, and a basic warehouse does not automatically change its size merely because a query is slow.

Sources: [Warehouse overview](https://docs.snowflake.com/en/user-guide/warehouses-overview), [Compute cost](https://docs.snowflake.com/en/user-guide/cost-understanding-compute).

## 17. Compute pools: a different compute resource

A **compute pool** contains one or more virtual machine nodes that run **Snowpark Container Services** services and jobs. This supports containerized applications and workloads needing a controlled runtime environment.

| Property | Meaning |
|---|---|
| Instance family | Node hardware profile; CPU/memory and GPU options depend on availability |
| `MIN_NODES` | Minimum configured node count |
| `MAX_NODES` | Upper limit for node scaling |
| Service instances | Application replicas scheduled onto nodes |

Snowflake can add nodes when service placement requires more capacity and remove unused capacity within configured bounds. Service replica scaling and pool node scaling are different controls: adding nodes alone does not rewrite an application to run in parallel.

A compute pool is an account-level resource. Available instance families can be inspected with `SHOW COMPUTE POOL INSTANCE FAMILIES`.

Source: [Working with compute pools](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/working-with-compute-pool).

## 18. Virtual warehouse versus compute pool versus serverless

| Question | Virtual warehouse | Compute pool | Serverless feature |
|---|---|---|---|
| Main purpose | SQL and supported Snowpark processing | Container services and jobs | Specific managed operations |
| Capacity control | Size and cluster configuration | Instance family and node bounds | Feature-specific settings |
| Typical example | Dashboard aggregation | Containerized inference service | Snowpipe file ingestion |
| Consumption model | Warehouse resources over time | Provisioned node resources over time | Feature-specific metering |

A container application may submit SQL to a warehouse. In that design, container execution and SQL execution consume different resources. A compute pool does not replace the warehouse used for those SQL queries.

Snowpark is a programming framework; using a Snowpark DataFrame does not by itself imply that a compute pool is required.

Sources: [Compute pools](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/working-with-compute-pool), [Understanding compute cost](https://docs.snowflake.com/en/user-guide/cost-understanding-compute).

## 19. Bringing data in and transforming it

| Pattern | Typical mechanism | Compute consideration |
|---|---|---|
| Batch files | Stage plus `COPY INTO` | User-managed warehouse |
| Continuous file loading | Snowpipe | Managed ingestion compute |
| Row-oriented streaming | Snowpipe Streaming | Dedicated ingestion model |

A **stage** identifies a location used for files. Internal stages are managed within Snowflake; external stages reference supported cloud storage. File format definitions tell Snowflake how to interpret files.

After loading, validate row counts, types, duplicates, and business rules before publishing trusted tables.

Source: [Data loading](https://docs.snowflake.com/en/user-guide/data-load-overview).

**Dynamic tables** maintain the result of a defined query using managed refreshes and a target lag. They can express transformation dependencies declaratively. Refresh mode and query eligibility affect whether work is incremental or full; target lag should be chosen according to freshness requirements and cost.

Source: [Dynamic tables](https://docs.snowflake.com/en/user-guide/dynamic-tables/overview).

## 20. Recovery: Time Travel and Fail-safe

**Time Travel** allows access to historical data within the configured retention period. Depending on the operation, it supports historical queries, cloning historical states, and restoring dropped objects.

The standard retention is one day. Eligible permanent objects on Enterprise Edition or higher can have retention up to 90 days. Table type, edition, and configuration determine the actual period; not every table has 90 days of history.

Source: [Time Travel](https://docs.snowflake.com/en/user-guide/data-time-travel).

**Fail-safe** is a separate, non-configurable seven-day recovery period for eligible permanent table data after Time Travel. It is a Snowflake-managed disaster recovery mechanism, not a self-service historical query feature. Temporary and transient tables do not have Fail-safe.

Source: [Fail-safe](https://docs.snowflake.com/en/user-guide/data-failsafe).

Retention is useful but consumes storage. Design recovery requirements together with retention and cost; suspending a warehouse does not suspend historical data retention.

## 21. Sharing, security, and responsibility

Secure Data Sharing lets a provider expose selected data objects to consumers. In the supported same-region sharing model, consumers query shared data without a separate full data copy and normally use their own compute. Cross-region or cross-cloud access can involve replication or other distribution mechanisms and costs.

Source: [Secure Data Sharing](https://docs.snowflake.com/en/user-guide/data-sharing-intro).

Access is controlled through privileges and roles. For example, an analyst may need `USAGE` on a warehouse, database, and schema, plus `SELECT` on a table. Permission to use compute does not automatically grant permission to read all data.

Use separate roles for administration, ingestion, transformation, and analysis. Give each role only the privileges its work requires. Managed infrastructure still leaves customers responsible for correct data access, data quality, and appropriate configuration.

Source: [Access control overview](https://docs.snowflake.com/en/user-guide/security-access-control-overview).

## 22. Understanding cost with worked examples

Major categories include storage, warehouse compute, serverless features, compute pools, and applicable data transfer charges. Cloud services can also contribute billed usage according to Snowflake's adjustment rules.

For a warehouse-only estimate:

```text
Credits = sum of (each cluster's hourly rate × its billable hours)
Money   = credits × your contracted price per credit
```

**Illustrative Gen1 calculations:**

| Scenario | Calculation | Credits |
|---|---|---:|
| Small, one cluster, 30 minutes | 2 × 0.5 | 1 |
| Medium, two clusters, 15 minutes each | 4 × 2 × 0.25 | 2 |
| Large completes a job in 6 minutes | 8 × 6/60 | 0.8 |
| Medium completes the same job in 15 minutes | 4 × 15/60 | 1 |

The last two rows assume measured runtimes; they are not performance predictions. Include startup minimums and idle runtime in real calculations. Credit prices depend on commercial terms; no fixed dollar rate is assumed here.

Sources: [Compute cost](https://docs.snowflake.com/en/user-guide/cost-understanding-compute), [Warehouse considerations](https://docs.snowflake.com/en/user-guide/warehouses-considerations).

## 23. Illustrative SQL: warehouse lifecycle

Read these statements as examples. Running them later in Snowflake requires an appropriately privileged role and can incur compute charges when a warehouse runs.

```sql
CREATE WAREHOUSE TRAINING_WH
  WAREHOUSE_TYPE = 'STANDARD'
  WAREHOUSE_SIZE = 'XSMALL'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE
  INITIALLY_SUSPENDED = TRUE;

USE WAREHOUSE TRAINING_WH;

-- Resize for subsequent work; benchmark before retaining the change.
ALTER WAREHOUSE TRAINING_WH SET WAREHOUSE_SIZE = 'SMALL';

SHOW WAREHOUSES;

ALTER WAREHOUSE TRAINING_WH SUSPEND;
```

Selecting a warehouse establishes session context. A later statement requiring compute can resume it. If it is already suspended, the final suspend statement may report that state.

Source: [Working with warehouses](https://docs.snowflake.com/en/user-guide/warehouses-tasks).

## 24. Illustrative SQL: multi-cluster warehouse and compute pool

The warehouse example requires Enterprise Edition or higher. It expresses cluster scaling, not automatic changes from Medium to Large.

```sql
CREATE WAREHOUSE REPORTING_WH
  WAREHOUSE_SIZE = 'MEDIUM'
  MIN_CLUSTER_COUNT = 1
  MAX_CLUSTER_COUNT = 3
  SCALING_POLICY = 'STANDARD'
  ENABLE_QUERY_ACCELERATION = FALSE
  AUTO_SUSPEND = 120
  AUTO_RESUME = TRUE
  INITIALLY_SUSPENDED = TRUE;
```

Query acceleration is explicitly disabled so this example describes warehouse cluster capacity alone. It is a separate feature with separate consumption.

Source: [Multi-cluster warehouses](https://docs.snowflake.com/en/user-guide/warehouses-multicluster).

```sql
-- Inspect the instance families supported by your account first.
SHOW COMPUTE POOL INSTANCE FAMILIES;

CREATE COMPUTE POOL TRAINING_POOL
  MIN_NODES = 1
  MAX_NODES = 2
  INSTANCE_FAMILY = CPU_X64_XS
  INITIALLY_SUSPENDED = TRUE;
```

This creates a pool definition, not a container service. The instance family is illustrative and must be available in the target account.

Source: [CREATE COMPUTE POOL](https://docs.snowflake.com/en/sql-reference/sql/create-compute-pool).

## 25. How to choose a scaling response

Use this teaching checklist with representative measurements:

| Observed symptom | Investigate | Possible response |
|---|---|---|
| One query is slow even without other users | Scanned data, joins, spilling, SQL shape | Improve SQL/layout; benchmark scale-up |
| Queries wait during dashboard peaks | Queue time and concurrency | Multi-cluster scale-out |
| Reporting competes with transformations | Workload overlap | Separate warehouses |
| Compute stays active after work ends | Idle runtime and suspension settings | Adjust auto-suspend |
| First query after resume is slower | Cold warehouse cache | Balance cache reuse against idle cost |
| Larger load warehouse brings little improvement | File count, file sizes, ingestion bottleneck | Improve load parallelism/layout |

Measure both runtime and credits. Start from the workload's latency, throughput, and freshness targets; there is no universally correct size for a given table size.

Sources: [Warehouse considerations](https://docs.snowflake.com/en/user-guide/warehouses-considerations), [Warehouse cache](https://docs.snowflake.com/en/user-guide/performance-query-warehouse-cache).

## 26. End-to-end retail scenario

This is a proposed design exercise, not a benchmark or required production configuration.

1. Land daily order files in a stage and load `RAW.ORDERS` using `INGEST_WH`.
2. Validate totals and reject invalid business records before publishing curated data.
3. Use `TRANSFORM_WH` to build a customer-by-day sales table.
4. Give analysts read access to the analytics schema and compute access to `BI_WH`.
5. If the transformation is slow in isolation, inspect its plan and compare warehouse sizes.
6. If dashboards queue at 9 AM, evaluate multi-cluster reporting capacity.
7. Suspend idle warehouses after each workload finishes.
8. Add a compute pool only if the design includes a container service, such as a custom inference API.

The stored sales data is independent of these compute lifecycles. Workload separation allows different capacity, scheduling, and cost decisions for ingestion, transformation, and reporting.

## 27. Review questions and answers

| Question | Answer |
|---|---|
| When was Snowflake founded? | 2012. |
| Where is Snowflake currently listed? | NYSE, ticker SNOW; verified 7 September 2026. |
| Does suspending a warehouse delete tables? | No. Persistent storage has an independent lifecycle. |
| Does X-Large mean a larger storage allowance? | No. It describes compute capacity. |
| What does pruning do? | Skips partitions that metadata shows are irrelevant. |
| What is the usual response to one resource-heavy query? | Investigate SQL and data layout; benchmark scale-up. |
| What addresses many queued analytical queries? | Consider scale-out and workload isolation. |
| Is a compute pool a SQL warehouse? | No. It runs container services and jobs. |
| Does a larger warehouse always save time or money? | No. Measure representative workloads. |
| Is Fail-safe the same as Time Travel? | No. Their access and recovery purposes differ. |

**Practice:** A team has one long-running transformation, 200 dashboard users arriving together, and idle compute overnight. Explain three separate changes you would evaluate, which metrics would justify them, and how each change affects cost.

**Suggested answer:** inspect and benchmark the transformation for scale-up; inspect reporting queues for multi-cluster capacity; configure suspension for overnight inactivity. Track query runtime, queue time, and metered credits, respectively.